In [ ]:
import asyncio
import base64
import os
import platform
import threading
import time
import uuid
from pathlib import Path
from typing import Annotated, Any, Callable, Literal

from pydantic import BaseModel, Field
from typing_extensions import TypedDict

from kagraph import END, START, StateGraph, add_messages, image_from_base64, prompt_llm
from kagraph.llms import load_llm
from kagraph.messages import AIMessage, AnyMessage, HumanMessage, ToolMessage
from kagraph.runtime import Runtime
from kagraph.tracing import trace
from playwright.async_api import Page, async_playwright

In [ ]:
DEFAULT_START_URL = "https://blog.google/innovation-and-ai/technology/developers-tools/kaggle-community-benchmarks/"
DEFAULT_SEARCH_URL = "https://duckduckgo.com"
DEFAULT_MODEL = os.getenv("WEB_VOYAGER_MODEL", "openai/gpt-5.5-2026-04-23")
DEFAULT_RECORDING_DIR = Path("artifacts") / "web_voyager"
EXAMPLE_TASK = "Summarize the content in this page."
EXAMPLE_MAX_STEPS = 8


_BROWSER_PAGES: dict[str, Page] = {}
_THREAD_LOCAL = threading.local()


def add_int(left: int | None = None, right: int | None = None) -> int:
    return (left or 0) + (right or 0)


MARK_PAGE_SCRIPT = r"""
(() => {
  if (window.__kagraphWebVoyagerInstalled) {
    return;
  }

  window.__kagraphWebVoyagerLabels = [];

  window.kagraphUnmarkPage = function () {
    for (const label of window.__kagraphWebVoyagerLabels) {
      label.remove();
    }
    window.__kagraphWebVoyagerLabels = [];
  };

  window.kagraphMarkPage = function () {
    window.kagraphUnmarkPage();

    const vw = Math.max(document.documentElement.clientWidth || 0, window.innerWidth || 0);
    const vh = Math.max(document.documentElement.clientHeight || 0, window.innerHeight || 0);

    let items = Array.from(document.querySelectorAll("*"))
      .map((element) => {
        const textualContent = element.textContent.trim().replace(/\s{2,}/g, " ");
        const elementType = element.tagName.toLowerCase();
        const ariaLabel = element.getAttribute("aria-label") || "";

        const rects = Array.from(element.getClientRects())
          .filter((bb) => {
            const centerX = bb.left + bb.width / 2;
            const centerY = bb.top + bb.height / 2;
            const elementAtCenter = document.elementFromPoint(centerX, centerY);
            return elementAtCenter === element || element.contains(elementAtCenter);
          })
          .map((bb) => {
            const rect = {
              left: Math.max(0, bb.left),
              top: Math.max(0, bb.top),
              right: Math.min(vw, bb.right),
              bottom: Math.min(vh, bb.bottom),
            };
            return {
              ...rect,
              width: rect.right - rect.left,
              height: rect.bottom - rect.top,
            };
          })
          .filter((rect) => rect.width > 0 && rect.height > 0);

        const area = rects.reduce((total, rect) => total + rect.width * rect.height, 0);
        const include =
          element.tagName === "INPUT" ||
          element.tagName === "TEXTAREA" ||
          element.tagName === "SELECT" ||
          element.tagName === "BUTTON" ||
          element.tagName === "A" ||
          element.onclick != null ||
          window.getComputedStyle(element).cursor === "pointer" ||
          element.tagName === "IFRAME" ||
          element.tagName === "VIDEO";

        return {
          element,
          include,
          area,
          rects,
          text: textualContent,
          type: elementType,
          ariaLabel,
        };
      })
      .filter((item) => item.include && item.area >= 20);

    items = items.filter(
      (item) => !items.some((other) => item.element.contains(other.element) && item !== other)
    );

    function colorForIndex(index) {
      const colors = ["#22c55e", "#3b82f6", "#f97316", "#a855f7", "#ef4444", "#14b8a6"];
      return colors[index % colors.length];
    }

    items.forEach((item, index) => {
      item.rects.forEach((bbox) => {
        const marker = document.createElement("div");
        const borderColor = colorForIndex(index);
        marker.style.outline = `2px dashed ${borderColor}`;
        marker.style.position = "fixed";
        marker.style.left = `${bbox.left}px`;
        marker.style.top = `${bbox.top}px`;
        marker.style.width = `${bbox.width}px`;
        marker.style.height = `${bbox.height}px`;
        marker.style.pointerEvents = "none";
        marker.style.boxSizing = "border-box";
        marker.style.zIndex = 2147483647;

        const label = document.createElement("span");
        label.textContent = String(index);
        label.style.position = "absolute";
        label.style.top = "-19px";
        label.style.left = "0";
        label.style.background = borderColor;
        label.style.color = "white";
        label.style.padding = "2px 4px";
        label.style.fontSize = "12px";
        label.style.borderRadius = "2px";
        marker.appendChild(label);

        document.body.appendChild(marker);
        window.__kagraphWebVoyagerLabels.push(marker);
      });
    });

    return items.flatMap((item) =>
      item.rects.map(({ left, top, width, height }) => ({
        x: left + width / 2,
        y: top + height / 2,
        type: item.type,
        text: item.text,
        ariaLabel: item.ariaLabel,
      }))
    );
  };

  window.__kagraphWebVoyagerInstalled = true;
})();
"""


class BBox(TypedDict):
    x: float
    y: float
    text: str
    type: str
    ariaLabel: str


ActionName = Literal["Click", "Type", "Scroll", "Wait", "GoBack", "Search", "Google", "ANSWER"]


class BrowserPrediction(BaseModel):
    thought: str = Field(description="Brief reason for the selected browser action.")
    action: ActionName = Field(description="The next browser action to execute.")
    args: list[str] = Field(
        default_factory=list,
        description=(
            "Action arguments. Click: [bbox_id]. Type: [bbox_id, text]. "
            "Scroll: [WINDOW or bbox_id, up or down]. Wait/GoBack/Search: []. "
            "ANSWER: [final answer]."
        ),
    )


class WebVoyagerInput(TypedDict, total=False):
    task: str
    start_url: str


class WebVoyagerState(TypedDict, total=False):
    task: str
    start_url: str
    bboxes: list[BBox]
    prediction: dict[str, Any]
    scratchpad: list[str]
    observation: str
    browser_actions: Annotated[int, add_int]
    final_answer: str
    messages: Annotated[list[AnyMessage], add_messages]


class WebVoyagerContext(TypedDict, total=False):
    browser_session_id: str
    max_browser_actions: int
    wait_ms: int
    search_url: str


WEB_VOYAGER_SYSTEM_PROMPT = """You are WebVoyager, a vision-enabled browser agent.

You receive a browser screenshot with numbered bounding boxes marking clickable or editable UI elements.
Use the screenshot, current URL, bounding box descriptions, and previous observations to complete the user's task.

Choose exactly one action:
- Click with args [bbox_id]
- Type with args [bbox_id, text]
- Scroll with args [WINDOW or bbox_id, up or down]
- Wait with args []
- GoBack with args []
- Search with args []
- ANSWER with args [final answer]

Rules:
- Use only bounding box ids shown in the current observation.
- Prefer Type on search boxes or form fields.
- Use Search when you need to return to the search engine.
- If a page is blocked by reCAPTCHA or an automated browsing challenge, use Search or GoBack instead of answering that you are blocked.
- Use ANSWER only when the user's task is complete.
- Return through the structured output schema only.
"""


def build_graph(llm: Any):
    graph = StateGraph(
        WebVoyagerState,
        input_schema=WebVoyagerInput,
        context_schema=WebVoyagerContext,
    )

    async def prepare(state: WebVoyagerInput, *, runtime: Runtime):
        page = _page(runtime)
        start_url = state.get("start_url") or DEFAULT_START_URL
        await page.goto(start_url, wait_until="domcontentloaded")
        await _settle_page(page, runtime)
        return {
            "task": state["task"],
            "start_url": start_url,
            "scratchpad": [],
            "browser_actions": 0,
            "messages": [HumanMessage(state["task"])],
        }

    async def agent(state: WebVoyagerState, *, runtime: Runtime):
        page = _page(runtime)
        screenshot_b64, bboxes = await _annotate_page(page)
        prompt = _build_agent_prompt(state, page.url, bboxes)
        image = image_from_base64(screenshot_b64, format="png")
        prediction = prompt_llm(
            llm,
            prompt,
            system=WEB_VOYAGER_SYSTEM_PROMPT,
            image=image,
            schema=BrowserPrediction,
            chat_name="KaGraph WebVoyager LLM",
        )
        if isinstance(prediction, dict):
            prediction_dict = BrowserPrediction.model_validate(prediction).model_dump()
        else:
            prediction_dict = prediction.model_dump()
        return {
            "bboxes": bboxes,
            "prediction": prediction_dict,
            "messages": [AIMessage(_format_prediction(prediction_dict))],
        }

    async def click_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _click(state, runtime)}

    async def type_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _type_text(state, runtime)}

    async def scroll_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _scroll(state, runtime)}

    async def wait_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _wait(state, runtime)}

    async def go_back_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _go_back(state, runtime)}

    async def google_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _go_to_search(runtime)}

    async def search_node(state: WebVoyagerState, *, runtime: Runtime):
        return {"observation": await _go_to_search(runtime)}

    def update_scratchpad(state: WebVoyagerState):
        prediction = state.get("prediction") or {}
        observation = state.get("observation", "")
        entry = (
            f"{len(state.get('scratchpad') or []) + 1}. "
            f"Thought: {prediction.get('thought', '')}\n"
            f"Action: {prediction.get('action')} {prediction.get('args', [])}\n"
            f"Observation: {observation}"
        )
        return {
            "scratchpad": [*(state.get("scratchpad") or []), entry],
            "browser_actions": 1,
            "messages": [ToolMessage(observation)],
        }

    def final_answer(state: WebVoyagerState):
        prediction = state.get("prediction") or {}
        args = prediction.get("args") or []
        answer = _dedupe_repeated_text(str(args[0]) if args else prediction.get("thought") or "Done.")
        return {"final_answer": answer, "messages": [AIMessage(answer)]}

    def max_steps_answer(state: WebVoyagerState, *, runtime: Runtime):
        max_steps = _max_browser_actions(runtime)
        answer = (
            f"I could not complete the task within {max_steps} browser action steps. "
            "Increase max_steps and run again."
        )
        return {"final_answer": answer, "messages": [AIMessage(answer)]}

    def route_after_agent(state: WebVoyagerState, runtime: Runtime):
        prediction = state.get("prediction") or {}
        action = prediction.get("action")
        if action == "ANSWER":
            return "final_answer"
        if int(state.get("browser_actions", 0)) >= _max_browser_actions(runtime):
            return "max_steps_answer"
        return action

    def route_after_scratchpad(state: WebVoyagerState, runtime: Runtime):
        if int(state.get("browser_actions", 0)) >= _max_browser_actions(runtime):
            return "max_steps_answer"
        return "agent"

    graph.add_node("prepare", prepare, input_schema=WebVoyagerInput)
    graph.add_node("agent", agent)
    graph.add_node("Click", click_node)
    graph.add_node("Type", type_node)
    graph.add_node("Scroll", scroll_node)
    graph.add_node("Wait", wait_node)
    graph.add_node("GoBack", go_back_node)
    graph.add_node("Search", search_node)
    graph.add_node("Google", google_node)
    graph.add_node("update_scratchpad", update_scratchpad)
    graph.add_node("final_answer", final_answer)
    graph.add_node("max_steps_answer", max_steps_answer)

    graph.add_edge(START, "prepare")
    graph.add_edge("prepare", "agent")
    graph.add_conditional_edges(
        "agent",
        route_after_agent,
        {
            "Click": "Click",
            "Type": "Type",
            "Scroll": "Scroll",
            "Wait": "Wait",
            "GoBack": "GoBack",
            "Search": "Search",
            "Google": "Google",
            "final_answer": "final_answer",
            "max_steps_answer": "max_steps_answer",
        },
    )
    for node_name in ["Click", "Type", "Scroll", "Wait", "GoBack", "Search", "Google"]:
        graph.add_edge(node_name, "update_scratchpad")
    graph.add_conditional_edges(
        "update_scratchpad",
        route_after_scratchpad,
        {
            "agent": "agent",
            "max_steps_answer": "max_steps_answer",
        },
    )
    graph.add_edge("final_answer", END)
    graph.add_edge("max_steps_answer", END)

    return graph.compile(name="web_voyager")


async def run_web_voyager(
    llm: Any,
    task: str,
    *,
    start_url: str = DEFAULT_START_URL,
    max_steps: int = 12,
    headless: bool = True,
    recording_dir: str | Path = DEFAULT_RECORDING_DIR,
    viewport: dict[str, int] | None = None,
    trace_name: str | None = "WebVoyager",
) -> dict[str, Any]:
    """Run WebVoyager and return the graph result plus recording paths.

    In Jupyter, call this with ``await run_web_voyager(llm, task)``.
    """

    if _should_run_playwright_in_thread():
        return await asyncio.to_thread(
            _run_async_in_playwright_loop,
            _run_web_voyager_current_loop,
            llm,
            task,
            start_url=start_url,
            max_steps=max_steps,
            headless=headless,
            recording_dir=recording_dir,
            viewport=viewport,
            trace_name=trace_name,
        )
    return await _run_web_voyager_current_loop(
        llm,
        task,
        start_url=start_url,
        max_steps=max_steps,
        headless=headless,
        recording_dir=recording_dir,
        viewport=viewport,
        trace_name=trace_name,
    )


async def _run_web_voyager_current_loop(
    llm: Any,
    task: str,
    *,
    start_url: str = DEFAULT_START_URL,
    max_steps: int = 12,
    headless: bool = True,
    recording_dir: str | Path = DEFAULT_RECORDING_DIR,
    viewport: dict[str, int] | None = None,
    trace_name: str | None = "WebVoyager",
) -> dict[str, Any]:
    recording_path = Path(recording_dir)
    recording_path.mkdir(parents=True, exist_ok=True)
    viewport = viewport or {"width": 1280, "height": 900}
    session_id = f"web-voyager-{uuid.uuid4().hex}"

    async with async_playwright() as playwright:
        browser = await playwright.chromium.launch(headless=headless)
        context = await browser.new_context(
            viewport=viewport,
            record_video_dir=str(recording_path),
            record_video_size=viewport,
        )
        page = await context.new_page()
        _BROWSER_PAGES[session_id] = page
        result: dict[str, Any] | None = None
        raw_video_path: Path | None = None
        try:
            app = build_graph(llm)
            run_context = {
                "browser_session_id": session_id,
                "max_browser_actions": max_steps,
                "wait_ms": 1000,
                "search_url": DEFAULT_SEARCH_URL,
            }
            invoke_kwargs = {
                "input": {"task": task, "start_url": start_url},
                "context": run_context,
                "chat_name": "KaGraph WebVoyager example",
                "recursion_limit": max(20, max_steps * 4 + 8),
            }
            if trace_name:
                with trace(trace_name):
                    result = await app.ainvoke(**invoke_kwargs)
            else:
                result = await app.ainvoke(**invoke_kwargs)
        finally:
            video = page.video
            await page.close()
            await context.close()
            await browser.close()
            _BROWSER_PAGES.pop(session_id, None)
            if video is not None:
                raw_video_path = Path(await video.path())

    if result is None:
        result = {}
    result["raw_video_path"] = str(raw_video_path) if raw_video_path else None
    result["video_path"] = str(raw_video_path) if raw_video_path else None
    result["video_format"] = raw_video_path.suffix.lstrip(".") if raw_video_path else None
    return result


def load_web_voyager_llm(model_id: str | None = None):
    return load_llm(model_id or DEFAULT_MODEL, support_structured_outputs=True)


def _should_run_playwright_in_thread() -> bool:
    if platform.system() != "Windows":
        return False
    if getattr(_THREAD_LOCAL, "in_playwright_loop", False):
        return False
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return False
    return "Proactor" not in type(loop).__name__


def _run_async_in_playwright_loop(
    async_fn: Callable[..., Any],
    *args: Any,
    **kwargs: Any,
) -> Any:
    policy_cls = getattr(asyncio, "WindowsProactorEventLoopPolicy", None)
    if policy_cls is not None:
        asyncio.set_event_loop_policy(policy_cls())

    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)
    _THREAD_LOCAL.in_playwright_loop = True
    try:
        return loop.run_until_complete(async_fn(*args, **kwargs))
    finally:
        _THREAD_LOCAL.in_playwright_loop = False
        try:
            loop.run_until_complete(loop.shutdown_asyncgens())
        finally:
            asyncio.set_event_loop(None)
            loop.close()


def _page(runtime: Runtime) -> Page:
    session_id = runtime.context.get("browser_session_id")
    if not session_id:
        raise RuntimeError("Missing browser_session_id in WebVoyager runtime context.")
    try:
        return _BROWSER_PAGES[str(session_id)]
    except KeyError as error:
        raise RuntimeError(f"No active Playwright page for browser session {session_id!r}.") from error


def _max_browser_actions(runtime: Runtime) -> int:
    return int(runtime.context.get("max_browser_actions", runtime.context.get("max_steps", 12)))


async def _annotate_page(page: Page) -> tuple[str, list[BBox]]:
    bboxes: list[BBox] = []
    for attempt in range(10):
        try:
            try:
                await page.wait_for_load_state("domcontentloaded", timeout=1000)
            except Exception:
                pass
            await page.evaluate(MARK_PAGE_SCRIPT)
            bboxes = await page.evaluate("kagraphMarkPage()")
            break
        except Exception as error:
            if attempt == 9:
                raise error
            await asyncio.sleep(0.5)
    screenshot = await page.screenshot(full_page=False)
    await page.evaluate("kagraphUnmarkPage()")
    return base64.b64encode(screenshot).decode("ascii"), bboxes


def _build_agent_prompt(state: WebVoyagerState, current_url: str, bboxes: list[BBox]) -> str:
    bbox_lines = []
    for index, bbox in enumerate(bboxes):
        label = bbox.get("ariaLabel") or bbox.get("text") or ""
        bbox_lines.append(f'{index} (<{bbox.get("type", "element")}/>): "{label[:300]}"')
    bbox_descriptions = "\n".join(bbox_lines) if bbox_lines else "No clickable/editable boxes detected."
    scratchpad = "\n\n".join(state.get("scratchpad") or []) or "No previous actions."
    return f"""User task:
{state["task"]}

Current URL:
{current_url}

Previous action observations:
{scratchpad}

Valid bounding boxes:
{bbox_descriptions}

Decide the next browser action from the screenshot and these descriptions."""


def _format_prediction(prediction: dict[str, Any]) -> str:
    return (
        f"Thought: {prediction.get('thought', '')}\n"
        f"Action: {prediction.get('action')}\n"
        f"Args: {prediction.get('args', [])}"
    )


def _dedupe_repeated_text(text: str) -> str:
    text = text.strip()
    if not text:
        return text
    midpoint = len(text) // 2
    if len(text) % 2 == 0 and text[:midpoint].strip() == text[midpoint:].strip():
        return text[:midpoint].strip()
    return text


def _prediction_args(state: WebVoyagerState) -> list[str]:
    args = (state.get("prediction") or {}).get("args") or []
    return [str(arg) for arg in args]


def _bbox(state: WebVoyagerState, bbox_id: str) -> BBox:
    try:
        index = int(bbox_id)
        return state["bboxes"][index]
    except Exception as error:
        raise ValueError(f"No bounding box with id {bbox_id!r}.") from error


async def _click(state: WebVoyagerState, runtime: Runtime) -> str:
    args = _prediction_args(state)
    if len(args) != 1:
        return f"Click failed: expected [bbox_id], got {args!r}."
    bbox = _bbox(state, args[0])
    page = _page(runtime)
    await page.mouse.click(bbox["x"], bbox["y"])
    await _settle_page(page, runtime)
    return f"Clicked bounding box {args[0]}."


async def _type_text(state: WebVoyagerState, runtime: Runtime) -> str:
    args = _prediction_args(state)
    if len(args) != 2:
        return f"Type failed: expected [bbox_id, text], got {args!r}."
    bbox = _bbox(state, args[0])
    text = args[1]
    page = _page(runtime)
    await page.mouse.click(bbox["x"], bbox["y"])
    select_all = "Meta+A" if platform.system() == "Darwin" else "Control+A"
    await page.keyboard.press(select_all)
    await page.keyboard.press("Backspace")
    await page.keyboard.type(text)
    await page.keyboard.press("Enter")
    await _settle_page(page, runtime)
    return f"Typed {text!r} into bounding box {args[0]} and submitted."


async def _scroll(state: WebVoyagerState, runtime: Runtime) -> str:
    args = _prediction_args(state)
    if len(args) != 2:
        return f"Scroll failed: expected [WINDOW or bbox_id, up or down], got {args!r}."
    target, direction = args
    amount = -500 if direction.lower() == "up" else 500
    page = _page(runtime)
    if target.upper() == "WINDOW":
        await page.evaluate("(amount) => window.scrollBy(0, amount)", amount)
        await _settle_page(page, runtime)
        return f"Scrolled window {direction}."
    bbox = _bbox(state, target)
    await page.mouse.move(bbox["x"], bbox["y"])
    await page.mouse.wheel(0, amount)
    await _settle_page(page, runtime)
    return f"Scrolled element {target} {direction}."


async def _wait(state: WebVoyagerState, runtime: Runtime) -> str:
    page = _page(runtime)
    await _settle_page(page, runtime, multiplier=2)
    return "Waited for the page."


async def _go_back(state: WebVoyagerState, runtime: Runtime) -> str:
    page = _page(runtime)
    await page.go_back(wait_until="domcontentloaded")
    await _settle_page(page, runtime)
    return f"Went back to {page.url}."


async def _go_to_search(runtime: Runtime) -> str:
    page = _page(runtime)
    search_url = runtime.context.get("search_url") or DEFAULT_SEARCH_URL
    await page.goto(str(search_url), wait_until="domcontentloaded")
    await _settle_page(page, runtime)
    return f"Navigated to {search_url}."


async def _settle_page(page: Page, runtime: Runtime, *, multiplier: int = 1) -> None:
    wait_ms = int(runtime.context.get("wait_ms", 1000)) * multiplier
    try:
        await page.wait_for_load_state("networkidle", timeout=wait_ms)
    except Exception:
        await page.wait_for_timeout(wait_ms)


async def main_async(
    task: str = EXAMPLE_TASK,
    *,
    model: str = DEFAULT_MODEL,
    start_url: str = DEFAULT_START_URL,
    max_steps: int = EXAMPLE_MAX_STEPS,
    recording_dir: str | Path = DEFAULT_RECORDING_DIR,
    trace_name: str | None = "WebVoyager",
) -> dict[str, Any]:
    llm = load_web_voyager_llm(model)
    started = time.time()
    result = await run_web_voyager(
        llm,
        task,
        start_url=start_url,
        max_steps=max_steps,
        recording_dir=recording_dir,
        trace_name=trace_name,
    )
    print("FINAL ANSWER:", result.get("final_answer"))
    print("VIDEO:", result.get("video_path"))
    print("RAW VIDEO:", result.get("raw_video_path"))
    print(f"ELAPSED: {time.time() - started:.1f}s")
    return result


def main() -> None:
    asyncio.run(main_async())


if __name__ == "__main__":
    main()


: 